<a href="https://colab.research.google.com/github/ddickson28/FPSO-BN/blob/Add-Node/FPSOBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 165.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 163.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires nump

In [1]:
#Import modules
import numpy as np #importing a module
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import itertools
from scipy.stats import truncnorm
from math import erf, sqrt

In [5]:
#1 Define function for calculating combinations
def generate_indexed_combinations(state_counts):

    # Create ranges for each variable: 0..n_states-1
    ranges = [range(n) for n in state_counts]

    # Compute Cartesian product
    combos = list(itertools.product(*ranges))

    # Convert to numpy array
    return np.array(combos, dtype=int)

#Example usage:
#state_counts = [3, 2, 3]  # Child: 3 states, Parent1: 2 states, Parent2: 3 states
#C_matrix = generate_indexed_combinations(state_counts)
#print(C_matrix)

In [14]:
"""Defining the child node probability table requires defining all of the parent state
combinations and then using a T-Normal distribution to create discrete bins
across the defined child states|parent combination"""

# Define further node weights as required, ensure consistent ordering and sum to
# 1
N1 = 0.6
N2 = 0.3
N3 = 0.1

#Define the numeric scores for the states of ea. node. Follows ordering of nodes
#above.

P_Repair=np.array([1,0])
P_Offload=np.array([1, 0.5, 0])
P_Detail=np.array([1, 0])
#P_Additional=np.array([1, 0.75, 0.5, 0.25, 0])

#Define the weighted scores for the node vectors
A = N1*P_Repair
B = N2*P_Offload
C = N3*P_Detail

print(A)
print(B)
print(C)

#Create pairwise sum w/ specific ordering. Vector 1 moves slowest, vector 2
#moves fastest. ***Read this code block, format to extend?***

Parent_combination = A[:, None, None] + B[None, :, None] + C[None, None, :]

print(Parent_combination)

#Takes two rows and flattens into a 1-D vector
mu = Parent_combination.reshape(-1)

print(mu)

#Calculating truncated normal from mu vector above and specficied variance

var = 0.15
std = np.sqrt(var)

#Define intervals for truncnorm 0, 1/3 2/3, 1. Consider automating for more
#states. Reflects Low, Med, High
bins = np.array([0.0, 0.33, 0.66, 1])

#Build truncnorm and calculate CDF for each bin interval

a = (0 - mu) / std
b = (1 - mu) / std

cdf_val = truncnorm.cdf(bins[:, None], a, b, loc=mu, scale=std)

prob_vectors = np.diff(cdf_val, axis=0).T
prob_vectors = prob_vectors[:,::-1]
sum_vectors=prob_vectors.sum(axis=1)

print(prob_vectors)
print(sum_vectors)

#***Need to build the order here, 1st from each column***
flat_prob = prob_vectors.T.flatten()

print(flat_prob)


[0.6 0. ]
[0.3  0.15 0.  ]
[0.1 0. ]
[[[1.   0.9 ]
  [0.85 0.75]
  [0.7  0.6 ]]

 [[0.4  0.3 ]
  [0.25 0.15]
  [0.1  0.  ]]]
[1.   0.9  0.85 0.75 0.7  0.6  0.4  0.3  0.25 0.15 0.1  0.  ]
[[0.62613985 0.29930876 0.07455139]
 [0.56460865 0.33319884 0.10219251]
 [0.53227902 0.3489688  0.11875218]
 [0.4656007  0.37664605 0.15775325]
 [0.43179169 0.38795543 0.18025288]
 [0.36474194 0.40419284 0.23106522]
 [0.24140247 0.40673644 0.35186109]
 [0.18911696 0.39285434 0.41802871]
 [0.16587534 0.38260118 0.45152348]
 [0.12543605 0.3567199  0.51784405]
 [0.10819835 0.34167178 0.55012987]
 [0.07931591 0.30885677 0.61182732]]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[0.62613985 0.56460865 0.53227902 0.4656007  0.43179169 0.36474194 0.24140247 0.18911696 0.16587534 0.12543605 0.10819835 0.07931591 0.29930876 0.33319884 0.3489688  0.37664605 0.38795543 0.40419284 0.40673644 0.39285434 0.38260118 0.3567199  0.34167178 0.30885677 0.07455139 0.10219251 0.11875218 0.15775325 0.18025288 0.23106522 0.35186109 

In [15]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define the variables (nodes). N1, N2, ... ,Nn. Child node last


PreviousRepair = variable.Variable('PreviousRepair', ['True','False']) #Variable Class inside the variable module, creates an objected called "PreviousRepair".
OffloadCycle = variable.Variable('OffloadCycles', ['High','Med','Low']) #Variable states are organised worst to best. e.g. previous repair True and offload cycles High is worst
DetailQuality = variable.Variable('DetailQuality', ['Good','Bad']) #Either a good quality detail or bad quality detail

CrackLocationInterest = variable.Variable('CrackLocationInterest', ['High','Med','Low'])


#2 Define the cpm of the variables from 1.

cpm_PreviousRepair = cpm.Cpm(
									[PreviousRepair], no_child=1,
									 C=np.array([[0],[1]], dtype=int),
									 p=np.array([0.5,0.5])
)

cpm_OffloadCycle = cpm.Cpm(
									[OffloadCycle], no_child=1,
									 C=np.array([[0],[1],[2]], dtype=int),
									 p=np.array([0.33,0.33,0.34])
)

#Added node
cpm_DetailQuality = cpm.Cpm(
									[DetailQuality], no_child=1,
									 C=np.array([[0],[1]], dtype=int),
									 p=np.array([0.5,0.5])
)

# Creates all possible child states given parents
state_counts = [3, 2, 3, 2]  # Child: 3 states, Parent1: 2 states, Parent2: 3 states parent3: 2 states
#Go to code section above for creating P_child flat

C_child= generate_indexed_combinations(state_counts)
print(C_child)

P_child=flat_prob
print(P_child)

cpm_CrackLocationInterest = cpm.Cpm(
									[CrackLocationInterest, OffloadCycle, PreviousRepair, DetailQuality], no_child=1,
									 C=C_child,
									 p=P_child
)

print(cpm_CrackLocationInterest)

[[0 0 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [0 0 1 1]
 [0 0 2 0]
 [0 0 2 1]
 [0 1 0 0]
 [0 1 0 1]
 [0 1 1 0]
 [0 1 1 1]
 [0 1 2 0]
 [0 1 2 1]
 [1 0 0 0]
 [1 0 0 1]
 [1 0 1 0]
 [1 0 1 1]
 [1 0 2 0]
 [1 0 2 1]
 [1 1 0 0]
 [1 1 0 1]
 [1 1 1 0]
 [1 1 1 1]
 [1 1 2 0]
 [1 1 2 1]
 [2 0 0 0]
 [2 0 0 1]
 [2 0 1 0]
 [2 0 1 1]
 [2 0 2 0]
 [2 0 2 1]
 [2 1 0 0]
 [2 1 0 1]
 [2 1 1 0]
 [2 1 1 1]
 [2 1 2 0]
 [2 1 2 1]]
[0.62613985 0.56460865 0.53227902 0.4656007  0.43179169 0.36474194 0.24140247 0.18911696 0.16587534 0.12543605 0.10819835 0.07931591 0.29930876 0.33319884 0.3489688  0.37664605 0.38795543 0.40419284 0.40673644 0.39285434 0.38260118 0.3567199  0.34167178 0.30885677 0.07455139 0.10219251 0.11875218 0.15775325 0.18025288 0.23106522 0.35186109 0.41802871 0.45152348 0.51784405 0.55012987 0.61182732]
<CPM representing P(CrackLocationInterest | OffloadCycles, PreviousRepair, DetailQuality) at 0x7e362a3547d0>
+-------------------------+-----+-----------------+-----------+
|   CrackLocationInterest [ ... 

In [11]:
import sys
import numpy as np

# Set print options to display the full arrays without truncation
np.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize)

print("Full combinations (C_child):")
print(C_child)

print("\nFull probabilities (P_child):")
print(P_child)

# Optionally, you can also print a DataFrame combining them for better readability
import pandas as pd

# Create column names for C_child, assuming order: CrackLocationInterest, OffloadCycles, PreviousRepair, DetailQuality
parent_vars = ['CrackLocationInterest', 'OffloadCycles', 'PreviousRepair', 'DetailQuality']
child_var = 'Probability'

df_cpm = pd.DataFrame(C_child, columns=parent_vars)
df_cpm[child_var] = P_child

print("\nFull Conditional Probability Table (DataFrame):")
# Display the DataFrame, pandas should handle display for the full table
print(df_cpm.to_string())

Full combinations (C_child):
[[0 0 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [0 0 1 1]
 [0 0 2 0]
 [0 0 2 1]
 [0 1 0 0]
 [0 1 0 1]
 [0 1 1 0]
 [0 1 1 1]
 [0 1 2 0]
 [0 1 2 1]
 [1 0 0 0]
 [1 0 0 1]
 [1 0 1 0]
 [1 0 1 1]
 [1 0 2 0]
 [1 0 2 1]
 [1 1 0 0]
 [1 1 0 1]
 [1 1 1 0]
 [1 1 1 1]
 [1 1 2 0]
 [1 1 2 1]
 [2 0 0 0]
 [2 0 0 1]
 [2 0 1 0]
 [2 0 1 1]
 [2 0 2 0]
 [2 0 2 1]
 [2 1 0 0]
 [2 1 0 1]
 [2 1 1 0]
 [2 1 1 1]
 [2 1 2 0]
 [2 1 2 1]]

Full probabilities (P_child):
[[9.76589404e-01]
 [9.26690644e-01]
 [8.78008098e-01]
 [7.11982635e-01]
 [5.95945713e-01]
 [3.42068966e-01]
 [4.16473928e-02]
 [8.38681938e-03]
 [3.29187649e-03]
 [4.00456519e-04]
 [1.26393572e-04]
 [1.08250616e-05]
 [2.34026514e-02]
 [7.32125717e-02]
 [1.21678748e-01]
 [2.85334296e-01]
 [3.97078238e-01]
 [6.21893206e-01]
 [6.40586568e-01]
 [4.22147308e-01]
 [3.08510500e-01]
 [1.36368287e-01]
 [8.36144051e-02]
 [2.77960699e-02]
 [7.94475104e-06]
 [9.67843949e-05]
 [3.13153661e-04]
 [2.68306927e-03]
 [6.97604834e-03]
 [3.60378278e-02]
 [3